# Collab Architecture — RFP Content Generator
Generates all text sections for a Collab Architecture RFP response using Claude Sonnet.
Style is modeled on Collab's actual written RFP submissions — direct, specific, no filler.

**Steps:**
1. Set your project details in the **Configuration** cell
2. Run all cells top to bottom
3. Review `collab_rfp_generated.json` and `collab_rfp_output.pdf`

In [ ]:
import os
import json
import anthropic
import pdfplumber
from datetime import datetime
from dotenv import load_dotenv
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, HRFlowable, PageBreak
)

load_dotenv("./.env")

## Configuration
Edit the variables below for each new RFP.

In [ ]:
# ── PROJECT DETAILS ───────────────────────────────────────────────────────────
RFP_TITLE   = "Design and Construction of New Community Recreation Center"
CLIENT_NAME = "Town of Windsor, Colorado"
RFP_NUMBER  = "RFP-2026-001"
DATE        = datetime.today().strftime("%B %d, %Y")   # or set manually: "April 14, 2026"

# ── FILE PATHS ────────────────────────────────────────────────────────────────
TEMPLATE_PDF = "/Users/brianpak/Desktop/Projects/Collab Architecture/RFP_Project_2/1_RFP Base Template Files/2026 Master Template_Facing Pages.pdf"
OUTPUT_JSON  = "./collab_rfp_generated.json"
OUTPUT_PDF   = "./collab_rfp_output.pdf"

print(f"RFP    : {RFP_TITLE}")
print(f"Client : {CLIENT_NAME}")
print(f"Number : {RFP_NUMBER}")
print(f"Date   : {DATE}")

## Style Guide
Distilled from Collab Architecture's actual written RFP submissions.
Used as the system prompt on every Claude call to enforce consistent voice.

In [ ]:
STYLE_GUIDE = """
You are writing professional architecture RFQ/RFP response content for Collab Architecture,
a Windsor, Colorado-based firm. Match their documented voice exactly.

VOICE & TONE:
- Direct, confident, technically specific — never generic or filler
- Professional and warm without being sentimental
- Every sentence earns its place; no throat-clearing, no restating context
- Use bold (**word or phrase**) sparingly to flag key differentiators and facts
- Never begin with "We are excited to" or "We are pleased to"

COVER LETTER STRUCTURE (5-6 paragraphs, from actual Collab submissions):
1. Open with a direct statement about what the project represents — its stakes and purpose
2. State Collab's qualifications for this specific scope; mention delivery method if known
3. Detail the scope of services Collab will provide; name specific work items
4. Cite relevant experience with specific evidence (**bold** dollar values, project types)
5. Name key team leads and their roles; note subconsultant relationships
6. Commitment statement + close (one tight sentence each)

SECTION HEADLINE STYLE:
- All caps, 5-8 words, outcome/action-oriented
- Examples from real Collab RFPs:
    "DESIGNING FUNCTIONAL SPACES THAT ENHANCE PERFORMANCE"
    "DESIGNING SPACES THAT BRING COMMUNITIES TOGETHER"

FIRM IDENTITY (always factually accurate):
- Motto: "Stop. Collaborate and Listen." — literal working method, not just a tagline
- Mission: "Bringing the power of collaborative design to create a stronger, better,
  and more sustainable community."
- Address: 9217 Eastman Park Dr, Windsor, CO 80550
- Phone: 970-292-7078 | Web: www.collabarchitects.com
- Services: full-service commercial architecture, planning, engineering coordination

TEAM (use real names and credentials):
- Jordan W. Lockner, AIA, NCARB — Founding Principal; Principal-in-Charge
  Awards: 2022 UC ENVD Young Designer Award, 2023 BizWest 40 Under 40,
  2024 CASA of Larimer County Corporate Partner of the Year, 2025 SMPS Colorado Firm Leader of the Year
- Emily Perkins, AIT — Project Designer / Job Captain
  Recent work: Town of Eaton Public Library, Weld County School District 6, faith-based clients
- Kevin Dorsey — Design Manager (30+ years construction & design experience)
  Education: Aims Community College + Colorado State University
- Michael Aller — QA/QC Manager
- Bryan Merritt — Project Manager

SUBCONSULTANTS (name when relevant):
- Bridgers & Paxton — MEP/AV/IT engineering
- Corbel Engineering — structural
- JVA Consulting Engineers — civil

WRITING RULES:
- "Collab" or "Collab Architecture" for the firm name (not exclusively "we")
- Cite specific Colorado project names and locations when giving experience evidence
- Time commitments are stated as percentages (30%, 40%)
- Bold only 2-4 phrases per section maximum
- Return ONLY the requested text — no headers, labels, or meta-commentary
"""

## Section Definitions
Each section maps to a prompt template and a `target_contains` snippet
(matching placeholder text in the IDML template for future injection).

In [ ]:
SECTIONS = {

    # ── COVER PAGE ────────────────────────────────────────────────────────────
    "cover_rfp_title": {
        "label": "Cover Page — RFP Title",
        "page": 1,
        "target_contains": "RFP TITLE",
        "max_tokens": 50,
        "prompt": (
            "Write a clean 3-6 word project title for this RFP: {rfp_title} for {client_name}. "
            "Title case. No period. Return ONLY the title."
        ),
    },
    "cover_client": {
        "label": "Cover Page — Entity/Client",
        "page": 1,
        "target_contains": "ENTITY / CLIENT",
        "max_tokens": 30,
        "prompt": (
            "Return just the client/entity name formatted for a proposal cover page: {client_name}. "
            "All caps. No extra words."
        ),
    },

    # ── COVER LETTER (Page 3) ─────────────────────────────────────────────────
    "cover_letter_re": {
        "label": "Cover Letter — Re: line",
        "page": 3,
        "target_contains": "Re: RFP #",
        "max_tokens": 40,
        "prompt": (
            "Write a Re: line for a formal cover letter. "
            "Format: 'Re: {rfp_number} | {rfp_title}'. Return ONLY the Re: line."
        ),
    },
    "cover_letter_salutation": {
        "label": "Cover Letter — Salutation",
        "page": 3,
        "target_contains": "To [Client Name]",
        "max_tokens": 25,
        "prompt": (
            "Write a formal salutation for a cover letter to the selection committee at "
            "{client_name}. Use the format: "
            "'To [Contact Title] and Members of the Selection Committee,' "
            "Return ONLY the salutation line."
        ),
    },
    "cover_letter_body": {
        "label": "Cover Letter — Body",
        "page": 3,
        "target_contains": "Turehenis",
        "max_tokens": 650,
        "prompt": """
Write the body of a professional cover letter for Collab Architecture responding to this RFP.

RFP: {rfp_title}
Client: {client_name}
RFP Number: {rfp_number}
Date: {date}
Signatory: Jordan W. Lockner, AIA, NCARB, Founding Principal

Write exactly 6 paragraphs following this structure:
1. What this project represents — its stakes and purpose for the client (open with the project, not the firm)
2. Collab's qualifications for this exact scope and delivery method
3. Specific services Collab will provide (design phases, engineering coordination, code compliance)
4. Relevant prior experience — **bold** specific project types, dollar values, or milestones that prove fitness
5. Key team: name Jordan Lockner's role; name subconsultants Bridgers & Paxton, Corbel Engineering, JVA and their roles
6. One-sentence commitment to the client's mission + one-sentence close

Tone: confident, specific, zero filler. Bold 2-3 key phrases.
Return ONLY the letter body. No salutation, no "Sincerely", no headers.
""",
    },
    "cover_letter_date": {
        "label": "Cover Letter — Date",
        "page": 3,
        "target_contains": "Month Day, Year",
        "max_tokens": 20,
        "prompt": (
            "Return this date formatted for a formal letter: {date}. "
            "Example: 'February 4, 2026'. Return ONLY the formatted date."
        ),
    },

    # ── FIRM QUALIFICATIONS (Pages 6-7) ───────────────────────────────────────
    "firm_headline": {
        "label": "Firm Qualifications — Page Headline",
        "page": 6,
        "target_contains": "DESIGNING SPACES THAT BRING COMMUNITIES TOGETHER",
        "max_tokens": 30,
        "prompt": (
            "Write a 5-8 word headline for Collab Architecture's firm qualifications page. "
            "Tailor it to this project type: {rfp_title} for {client_name}. "
            "ALL CAPS. No period. Outcome-oriented (what does Collab design or deliver). "
            "Return ONLY the headline."
        ),
    },
    "who_we_are": {
        "label": "Firm Qualifications — Who We Are",
        "page": 6,
        "target_contains": "Our team brings specialized",
        "max_tokens": 200,
        "prompt": """
Write the 'WHO WE ARE' narrative for Collab Architecture's firm qualifications section.
This is for: {rfp_title} submitted to {client_name}.

Write 2 short paragraphs:
Para 1: Describe Collab as a Windsor-based firm, one of the fastest growing in Northern Colorado
and the Front Range. Driven by collaborative spirit. Reference the motto "Stop. Collaborate and Listen."
as the actual working method — curiosity, humility, honest dialogue, no pre-packaged solutions.

Para 2: Describe the firm's qualifications and resources for this specific project type —
full-service commercial architecture, multi-firm interdisciplinary team.
Name relevant subconsultants (Bridgers & Paxton, Corbel Engineering, JVA) and the specific
expertise they add for {rfp_title}.

Return ONLY the two paragraphs.
""",
    },
    "unique_team_knowledge": {
        "label": "Firm Qualifications — Unique Knowledge of Key Team Members",
        "page": 6,
        "target_contains": "Are we close to the project site",
        "max_tokens": 250,
        "prompt": """
Write the 'UNIQUE KNOWLEDGE OF KEY TEAM MEMBERS' narrative for Collab Architecture.
Project: {rfp_title} for {client_name}.

Write one short paragraph per team member using this format:
**Name** of Collab [role description relevant to this project].

Include these four in this order:
1. Jordan Lockner — leads multidisciplinary teams; relevant active-facility or technical project experience
2. Emily Perkins — rehabilitation and expansion of occupied facilities, structural mods, HVAC, additions
3. Kevin Dorsey — Design Manager, 30+ years construction/design, public facility expertise across Colorado
4. Michael Aller — QA/QC Manager, quality control and compliance review

Tailor each description to the specific demands of {rfp_title}.
Return ONLY the paragraphs, no section header.
""",
    },
    "on_site_presence": {
        "label": "Firm Qualifications — On-Site Presence",
        "page": 6,
        "target_contains": "Are we close to the project",
        "max_tokens": 90,
        "prompt": (
            "Write 2-3 sentences for the 'ON-SITE PRESENCE' callout box for Collab Architecture "
            "(Windsor, CO — 9217 Eastman Park Dr). "
            "Describe proximity to the project for {rfp_title} at {client_name}, "
            "availability for site visits, and responsiveness. "
            "Open with: 'When needs arise, [client shortname] needs a project manager and design person to rely on.' "
            "Return ONLY the callout text."
        ),
    },

    # ── OUR TEAM PAGE ─────────────────────────────────────────────────────────
    "team_experience_narrative": {
        "label": "Our Team — Experience on Projects as a Team",
        "page": 7,
        "target_contains": "The Collab team members on this project",
        "max_tokens": 160,
        "prompt": """
Write the 'EXPERIENCE ON PROJECTS AS A TEAM' paragraph for Collab Architecture's team page.
Project: {rfp_title} for {client_name}.

3-4 sentences covering:
- Collab team members work together daily and have designed similar projects throughout the firm's existence
- Collaboration — with each other and with clients — is where they excel and is a key benefit of the firm
- They partner with trusted subconsultants to bring specialized expertise for this specific project
- They bring existing workflow efficiencies and rapport to the proposed interdisciplinary team

Tailor any project-specific language to {rfp_title}.
Return ONLY the paragraph.
""",
    },
    "subconsultants_narrative": {
        "label": "Our Team — Subconsultants",
        "page": 7,
        "target_contains": "Our team is supported by",
        "max_tokens": 220,
        "prompt": """
Write the 'SUBCONSULTANTS' narrative for Collab Architecture's team page.
Project: {rfp_title} for {client_name}.

Write one intro sentence, then one short paragraph per subconsultant:

Intro: Our team is supported by a trusted group of subconsultants who bring specialized
technical expertise essential to the successful delivery of this project.

**JVA Consulting Engineers** — civil design services relevant to {rfp_title}
**Corbel Engineering** — structural design; balance structural upgrades with building integrity
**Bridgers & Paxton** — MEP and AV/IT; energy-efficient systems, lighting, security, HVAC

Each subconsultant paragraph should be 2 sentences, specific to what {rfp_title} demands.
Return ONLY the narrative. No section header.
""",
    },

    # ── TEAM BIOS (Pages 8-10) ────────────────────────────────────────────────
    "bio_lockner": {
        "label": "Team Bio — Jordan Lockner (Principal-in-Charge)",
        "page": 8,
        "target_contains": "Insert Bio",
        "max_tokens": 180,
        "prompt": """
Write the bio paragraph for Jordan W. Lockner, AIA, NCARB, Founding Principal of Collab Architecture.
His project role: Principal-in-Charge for {rfp_title}.

3-4 sentences covering:
- His belief that good architecture must stem from and support the community it exists within
- His ability to listen, understand, and collaborate while coordinating the design process and team
- Successful outcomes on projects of all sizes — large public projects to small business owners
- Well-versed in public projects throughout Northern Colorado; recognized leadership:
  2022 UC ENVD Young Designer Award, 2023 BizWest 40 Under 40, 2025 SMPS Colorado Firm Leader of the Year

Write in third person. Specific, no filler.
Return ONLY the bio paragraph.
""",
    },
    "bio_perkins": {
        "label": "Team Bio — Emily Perkins (Job Captain)",
        "page": 9,
        "target_contains": "Insert Bio",
        "max_tokens": 160,
        "prompt": """
Write the bio paragraph for Emily Perkins, AIT, Project Designer / Job Captain at Collab Architecture.
Her project role: Job Captain for {rfp_title}.

3-4 sentences covering:
- Leads projects from concept to completion with precision and creativity
- Passion for designing educational and community spaces that blend functionality,
  sustainability, and aesthetic appeal
- Recent work: Town of Eaton Public Library, Weld County School District 6, faith-based clients
- Spaces centered on learning, connection, and shared purpose; strong technical knowledge;
  fosters collaborative and inclusive environments

Write in third person. Specific, no filler.
Return ONLY the bio paragraph.
""",
    },
    "bio_dorsey": {
        "label": "Team Bio — Kevin Dorsey (Technical Design Manager)",
        "page": 10,
        "target_contains": "Insert Bio",
        "max_tokens": 160,
        "prompt": """
Write the bio paragraph for Kevin Dorsey, Design Manager at Collab Architecture.
His project role: Technical Design Manager for {rfp_title}.

3-4 sentences covering:
- 30+ years of construction and design experience; started as a craftsman and superintendent
- Degrees from Aims Community College and Colorado State University — combines field knowledge
  with technical expertise
- Designed wide range of public facilities across Colorado: K-12 schools, higher ed buildings,
  recreation centers, community hubs
- Responsible for project planning, construction documents, job-site observation, regulatory coordination

Write in third person. Specific, no filler.
Return ONLY the bio paragraph.
""",
    },

    # ── PROJECT EXPERIENCE ────────────────────────────────────────────────────
    "project_desc_1": {
        "label": "Project Experience — Description 1",
        "page": 16,
        "target_contains": "Mustibus eum iditatio",
        "max_tokens": 280,
        "prompt": """
Write a 3-paragraph project description for a Collab Architecture portfolio entry.
This entry should demonstrate experience directly relevant to: {rfp_title}.

Para 1: Project overview — the client's challenge and what was designed.
Use a real-sounding Colorado public-sector project (municipality, county, or state agency).
Include approximate construction value and location (City, CO).

Para 2: Design approach — how Collab addressed technical complexity, active operations,
phasing, stakeholder input, and code compliance specific to this project type.

Para 3: Outcome and lasting impact — what the completed facility delivers for its users
and why the design decisions will hold up long-term.

Write in third person past tense. Specific. Bold 1-2 key facts.
Return ONLY the three paragraphs.
""",
    },
    "project_desc_2": {
        "label": "Project Experience — Description 2",
        "page": 17,
        "target_contains": "Bores esendemos debitin",
        "max_tokens": 220,
        "prompt": """
Write a 2-paragraph project description for a second Collab Architecture portfolio project.
Also relevant to: {rfp_title}, but a different Colorado municipality and building type than the first entry.

Para 1: Project overview — client, challenge, scope, location (different city/county than previous).
Para 2: Design solution, notable technical or community challenge overcome, and outcome.

Write in third person past tense. Specific. Bold 1 key fact.
Return ONLY the two paragraphs.
""",
    },

    # ── PROJECT APPROACH ──────────────────────────────────────────────────────
    "project_approach": {
        "label": "Project Approach — Intro Narrative",
        "page": 22,
        "target_contains": "XX anticipates completing",
        "max_tokens": 220,
        "prompt": """
Write 2 paragraphs introducing Collab Architecture's project approach for {rfp_title}
submitted to {client_name}.

Para 1: Describe the overall design and delivery philosophy Collab brings to this project —
comprehensive evaluation of existing conditions, targeted rehabilitation, code compliance,
and how design decisions will support long-term operational performance.

Para 2: Describe the phased design process Collab will follow — from pre-design through
construction documents and administration. Note that Collab will refine the schedule
upon contract award in close coordination with {client_name}.

Specific, no filler. No 'XX' placeholders.
Return ONLY the two paragraphs.
""",
    },

    # ── PROJECT SCHEDULE ──────────────────────────────────────────────────────
    "schedule_project_name": {
        "label": "Project Schedule — Header Bar",
        "page": 24,
        "target_contains": "INSERT RFP TITLE OR PROJECT NAME",
        "max_tokens": 20,
        "prompt": (
            "Return a short 3-5 word project name for the schedule header bar for: {rfp_title}. "
            "ALL CAPS. Return ONLY the project name."
        ),
    },

    # ── EQUITY, DIVERSITY & INCLUSION (Page 30) ───────────────────────────────
    "edi_narrative": {
        "label": "EDI — Narrative",
        "page": 30,
        "target_contains": "Insert EDI",
        "max_tokens": 220,
        "prompt": """
Write the Equity, Diversity, and Inclusion narrative for Collab Architecture's RFP response.
Project: {rfp_title} for {client_name}.

2 paragraphs:
Para 1: How Collab embeds equity and inclusion into its design process — diverse stakeholder
engagement, accessible design, community-centered programming, and inclusive decision-making.

Para 2: Internal firm commitment — diverse hiring, mentorship (ACE Mentor), community board
service (Northern Colorado Unify, CASA of Larimer County), and how this internal culture
directly shapes better design outcomes for public clients.

Tone: genuine, specific, not performative.
Return ONLY the two paragraphs.
""",
    },

    # ── WORK LOCATION (Page 32) ───────────────────────────────────────────────
    "work_location": {
        "label": "Work Location — Narrative",
        "page": 32,
        "target_contains": "Are we close to the project site",
        "max_tokens": 130,
        "prompt": (
            "Write 3-4 sentences for the Work Location section of Collab Architecture's RFP response. "
            "Project: {rfp_title} for {client_name}. "
            "Cover: office location (Windsor, CO, 9217 Eastman Park Dr), proximity to project site, "
            "availability for on-site presence and rapid response, established Front Range project delivery. "
            "Tone: confident and practical. Return ONLY the sentences."
        ),
    },
}

## Helper Functions

In [ ]:
def extract_template_context(pdf_path, max_chars=4000):
    """Extract text from the IDML template PDF to give Claude layout context."""
    if not os.path.exists(pdf_path):
        print(f"  (template PDF not found at '{pdf_path}' — skipping context)")
        return ""
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages[:10]):
            t = page.extract_text()
            if t:
                chunks.append(f"[Page {i+1}]\n{t.strip()}")
    full = "\n\n".join(chunks)
    return full[:max_chars]

In [ ]:
def generate_all(client, rfp_title, client_name, date, rfp_number):
    """Generate all sections using Claude Sonnet with the STYLE_GUIDE system prompt."""
    generated = {}
    vars = {
        "rfp_title":   rfp_title,
        "client_name": client_name,
        "date":        date,
        "rfp_number":  rfp_number,
    }

    print(f"Generating {len(SECTIONS)} sections with Claude Sonnet...\n")

    for key, section in SECTIONS.items():
        print(f"  -> {section['label']}...", end=" ", flush=True)
        prompt = section["prompt"].format(**vars).strip()

        msg = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=section["max_tokens"],
            system=STYLE_GUIDE,
            messages=[{"role": "user", "content": prompt}]
        )
        text = msg.content[0].text.strip()
        generated[key] = {
            "label":           section["label"],
            "page":            section["page"],
            "target_contains": section["target_contains"],
            "generated_text":  text,
        }
        print(f"done  ({len(text)} chars)")

    return generated

In [ ]:
def build_output_pdf(generated, rfp_title, client_name, date, output_path):
    """Build a review PDF with all generated sections."""
    doc = SimpleDocTemplate(
        output_path,
        pagesize=letter,
        leftMargin=1*inch, rightMargin=1*inch,
        topMargin=1*inch,  bottomMargin=1*inch,
    )

    styles = getSampleStyleSheet()
    teal = colors.HexColor("#4A9FA5")
    dark = colors.HexColor("#1a1a2e")
    mid  = colors.HexColor("#2d2d2d")

    title_style   = ParagraphStyle("DocTitle", parent=styles["Title"],
                        fontSize=20, textColor=dark, spaceAfter=4)
    sub_style     = ParagraphStyle("Sub", parent=styles["Normal"],
                        fontSize=10, textColor=teal, spaceAfter=12)
    heading_style = ParagraphStyle("SectionH", parent=styles["Heading2"],
                        fontSize=12, textColor=teal, spaceBefore=16, spaceAfter=4)
    body_style    = ParagraphStyle("Body", parent=styles["Normal"],
                        fontSize=10, leading=16, textColor=mid, spaceAfter=8)
    meta_style    = ParagraphStyle("Meta", parent=styles["Normal"],
                        fontSize=8, textColor=colors.grey, spaceAfter=4, leftIndent=8)

    story = []

    # Header
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph("COLLAB ARCHITECTURE", sub_style))
    story.append(Paragraph(f"Response to RFP: {rfp_title}", title_style))
    story.append(Paragraph(f"Client: {client_name}  |  {date}", sub_style))
    story.append(HRFlowable(width="100%", thickness=2, color=teal))
    story.append(Spacer(1, 0.2*inch))

    # Each section
    for key, data in generated.items():
        content = data.get("generated_text", "(not generated)")
        story.append(Paragraph(data["label"].upper(), heading_style))
        story.append(Paragraph(
            f"Template page: {data['page']}  |  Target: \"{data['target_contains']}\"",
            meta_style
        ))
        story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#cccccc")))
        story.append(Spacer(1, 4))

        for line in content.split("\n"):
            line = line.strip()
            if not line:
                story.append(Spacer(1, 4))
            else:
                # Strip markdown bold markers for PDF rendering
                line = line.replace("**", "<b>", 1).replace("**", "</b>", 1)
                story.append(Paragraph(line, body_style))

        story.append(Spacer(1, 0.15*inch))

    doc.build(story)

## Run — Generate All Sections

In [ ]:
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise EnvironmentError("ANTHROPIC_API_KEY not set. Add it to your .env file or set it in the shell.")

client = anthropic.Anthropic(api_key=api_key)
print("Client ready.")

In [ ]:
print(f"Reading template: {TEMPLATE_PDF}")
template_context = extract_template_context(TEMPLATE_PDF)
if template_context:
    print(f"  {len(template_context)} chars of context extracted")
else:
    print("  Template PDF not found — continuing without layout context")

In [ ]:
generated = generate_all(client, RFP_TITLE, CLIENT_NAME, DATE, RFP_NUMBER)

## Save Outputs

In [ ]:
output_data = {
    "rfp_title":    RFP_TITLE,
    "client_name":  CLIENT_NAME,
    "date":         DATE,
    "rfp_number":   RFP_NUMBER,
    "generated_at": datetime.now().isoformat(),
    "model":        "claude-sonnet-4-6",
    "sections":     generated,
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"Saved JSON: {OUTPUT_JSON}")

In [ ]:
print(f"Building PDF: {OUTPUT_PDF}")
build_output_pdf(generated, RFP_TITLE, CLIENT_NAME, DATE, OUTPUT_PDF)
print(f"Saved PDF:  {OUTPUT_PDF}")

## Preview

In [ ]:
# Print a preview of every section
for key, data in generated.items():
    print(f"\n{'='*60}")
    print(f"  {data['label'].upper()}  (p.{data['page']})")
    print(f"{'='*60}")
    preview = data["generated_text"][:400]
    if len(data["generated_text"]) > 400:
        preview += "..."
    print(preview)